In [1]:
import gzip
import random
import pandas as pd
import xml.etree.ElementTree as ET

from tblib.decorators import return_error

In [56]:
# net xml 파싱하여 edge 목록 수집
net_file = './env-ny/ny/osm.net.xml'
net_tree = ET.parse(net_file)
net_root = net_tree.getroot()

In [57]:
valid_edges = set()
for edge_elem in net_root.findall('edge'):
    edge_id = edge_elem.get('id')
    if edge_id is not None:
        valid_edges.add(edge_id)

In [58]:
# person XML 파싱
person_file = './env-ny/ny/osm.person.xml'
person_tree = ET.parse(person_file)
person_root = person_tree.getroot()

In [59]:
# 유효하지 않은 승객 제거
persons_to_remove = []
for person_elem in person_root.findall('person'):
    ride_elem = person_elem.find('ride')
    if ride_elem is None:
        # ride 정보가 없으면 제거 대상
        persons_to_remove.append(person_elem)
        continue

    from_edge = ride_elem.get('from')
    to_edge = ride_elem.get('to')

    # from_edge 나 to_edge 둘 중 하나라도 net 파일에 없으면 제거 목록에 추가
    if (from_edge not in valid_edges) or (to_edge not in valid_edges):
        persons_to_remove.append(person_elem)

# 제거할 노드를 person_root에서 삭제
for p in persons_to_remove:
    person_root.remove(p)

# 새 XML 파일로 저장
filtered_person_file = "osm.person.xml"
person_tree.write(filtered_person_file, encoding='utf-8', xml_declaration=True)

print(f'Removed {len(persons_to_remove)} person nodes and made {filtered_person_file}.')

Removed 0 person nodes and made osm.person.xml.


In [60]:
person_file = 'osm.person.xml'
person_tree = ET.parse(person_file)
person_root = person_tree.getroot()

all_persons = person_root.findall('person')
print("Total persons #: ", len(all_persons))

Total persons #:  9615


In [ ]:
net_file = './env-ny/ny/osm.net.xml'
edges = []

try:
    with gzip.open(net_file, 'rb') as f:
        tree = ET.parse(f)
except OSError:
    tree = ET.parse(net_file)

net_root = tree.getroot()

for edge_elem in net_root.findall('edge'):
    if edge_elem.get('function') == 'internal':
        continue

    edge_id = edge_elem.get('id')
    if edge_id is None:
        continue

    skip_edge = False
    for lane_elem in edge_elem.findall('lane'):
        disallow_val = lane_elem.get('disallow') or ''
        allow_val = lane_elem.get('allow') or ''
        if 'taxi' in disallow_val.split() or 'bicycle' in allow_val.split():
            skip_edge = True
            break

    if not skip_edge:
        edges.append(edge_id)

print(f"Total edges #: {len(edges)}")

# routes 루트 엘리먼트 생성
routes_root = ET.Element('routes')

# vType 예시 추가 (taxi 차량 타입)
vtype_elem = ET.SubElement(routes_root, 'vType', {'id': 'taxi', 'vClass': 'taxi'})

# 차량 생성
num_vehicles = 10
departure_time = 806400
for i in range(num_vehicles):
    # vehicle 태그 만들기
    vehicle_elem = ET.SubElement(routes_root,
                                 'vehicle', {'depart': str(departure_time),
                                             'id': f'taxi_{i}',
                                             'line': 'taxi',
                                             'personNumber': '3',
                                             'type': 'taxi'})

    param_elem = ET.SubElement(vehicle_elem, 'param', {'key': 'has.taxi.device',
                                                       'value': 'true'})

    random_edge = random.choice(edges)
    route_elem = ET.SubElement(vehicle_elem, 'route', {'edges': random_edge})

# filtered_routes_file = 'env/env-ny/ny/osm.rou.xml'
tree = ET.ElementTree(routes_root)
# tree.write(filtered_routes_file, encoding='utf-8', xml_declaration=True)

print("Completed!")